# import Libraries

In [35]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import zscore
%matplotlib inline

#  Read Data

In [36]:
weather= pd.read_csv("data/processed_data_weather.csv")
charging=pd.read_csv("data/processed_data_charging.csv")

# merge Data

In [37]:
weather['timestamp'] = pd.to_datetime(weather['timestamp'])
weather['timestamp'].describe()
#the data start form 2018-01-01 08:53:00 to 2021-01-01 07:53:00

count                            29244
mean     2019-06-29 23:51:42.252770048
min                2018-01-01 08:53:00
25%                2018-10-06 05:38:00
50%                2019-06-24 00:23:00
75%                2020-03-25 20:08:00
max                2021-01-01 07:53:00
Name: timestamp, dtype: object

In [38]:
charging['connectionTime'] = pd.to_datetime(charging['connectionTime'])
charging['connectionTime'].describe()
#the data start form 2018-04-25 11:08:04+00:00 to 2021-09-14 05:43:39+00:00
#There is missing weather data from January 1, 2021, 7:53 AM to September 14, 2021, 5:43:39 AM. 

count                            66423
mean     2019-08-06 22:33:04.837872128
min                2018-04-25 11:08:04
25%         2018-11-13 14:23:23.500000
50%                2019-06-14 14:33:06
75%                2020-01-08 03:38:57
max                2021-09-14 05:43:39
Name: connectionTime, dtype: object

In [39]:
weather.set_index('timestamp', inplace=True)

In [40]:
start_date = '2020-01-01 07:53:05'
end_date = '2020-09-14 05:43:39'
reference_time=weather.loc[start_date:end_date]
reference_time.index=reference_time.index+pd.DateOffset(years=1)
reference_time.head(25)

,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation
timestamp,,,,,,
2021-01-01 08:53:00,9.0,33.0,Fair,989.77,11.0,0.0
2021-01-01 09:53:00,11.0,33.0,Fair,990.10,6.0,0.0
2021-01-01 10:53:00,9.0,29.0,Partly Cloudy,990.10,0.0,0.0
2021-01-01 11:53:00,8.0,29.0,Partly Cloudy,989.11,0.0,0.0
2021-01-01 12:53:00,7.0,29.0,Partly Cloudy,989.77,0.0,0.0
2021-01-01 13:53:00,7.0,33.0,Fair,989.44,0.0,0.0
2021-01-01 14:53:00,7.0,34.0,Fair,989.44,6.0,0.0
2021-01-01 15:53:00,8.0,34.0,Fair,989.77,7.0,0.0
2021-01-01 16:53:00,12.0,30.0,Partly Cloudy,990.76,9.0,0.0


In [41]:
weather_full=pd.concat([weather,reference_time])
weather_full.index

DatetimeIndex(['2018-01-01 08:53:00', '2018-01-01 09:53:00',
               '2018-01-01 10:53:00', '2018-01-01 11:53:00',
               '2018-01-01 12:53:00', '2018-01-01 13:53:00',
               '2018-01-01 14:53:00', '2018-01-01 15:53:00',
               '2018-01-01 16:53:00', '2018-01-01 17:53:00',
               ...
               '2021-09-13 20:02:00', '2021-09-13 20:53:00',
               '2021-09-13 21:53:00', '2021-09-13 22:53:00',
               '2021-09-13 23:53:00', '2021-09-14 00:53:00',
               '2021-09-14 01:53:00', '2021-09-14 02:53:00',
               '2021-09-14 03:53:00', '2021-09-14 04:53:00'],
              dtype='datetime64[ns]', name='timestamp', length=36070, freq=None)

In [42]:
charging['time']=charging['connectionTime']
charging['time']=pd.to_datetime(charging['time'])
charging=charging.set_index('time')
print(charging.index)
#This time includes a timezone(UTC).

DatetimeIndex(['2020-01-02 13:08:54', '2020-01-02 13:36:50',
               '2020-01-02 13:56:35', '2020-01-02 13:59:58',
               '2020-01-02 14:00:01', '2020-01-02 14:00:13',
               '2020-01-02 14:09:14', '2020-01-02 14:17:32',
               '2020-01-02 14:25:38', '2020-01-02 14:27:40',
               ...
               '2019-07-31 14:45:29', '2019-07-31 14:46:58',
               '2019-07-31 14:48:11', '2019-07-31 14:50:02',
               '2019-07-31 14:50:17', '2019-07-31 18:08:04',
               '2019-07-31 18:40:41', '2019-07-31 19:04:40',
               '2019-07-31 19:19:47', '2019-07-31 19:21:47'],
              dtype='datetime64[ns]', name='time', length=66423, freq=None)


In [43]:
charging['timezone'].unique()

array(['America/Los_Angeles'], dtype=object)

In [44]:

#since all the data is from the city of Los Angeles,timezone can be removed.
charging.index = charging.index.tz_localize(None)  
weather_full.index = weather_full.index.tz_localize(None)

In [45]:
charging = charging.sort_values(by='time')
weather_full = weather_full.sort_values(by='timestamp')
processed_data_all= pd.merge_asof(charging.reset_index(), weather_full.reset_index(), left_on='time', right_on='timestamp', direction='nearest')
print(processed_data_all)

                     time  Unnamed: 0                        id  \
0     2018-04-25 11:08:04           0  5bc90cb9f9af8b0d7fe77cd2   
1     2018-04-25 13:45:10           1  5bc90cb9f9af8b0d7fe77cd3   
2     2018-04-25 13:45:50           2  5bc90cb9f9af8b0d7fe77cd4   
3     2018-04-25 14:37:06           3  5bc90cb9f9af8b0d7fe77cd5   
4     2018-04-25 14:40:34           4  5bc90cb9f9af8b0d7fe77cd6   
...                   ...         ...                       ...   
66418 2021-09-13 22:33:07        3031  61550519f9af8b76960e169c   
66419 2021-09-13 23:11:12        3032  61550519f9af8b76960e169d   
66420 2021-09-14 01:08:16        5874  6155053bf9af8b76960e16d0   
66421 2021-09-14 01:52:37        3033  61550519f9af8b76960e169e   
66422 2021-09-14 05:43:39        5875  6155053bf9af8b76960e16d1   

           connectionTime       disconnectTime     doneChargingTime  \
0     2018-04-25 11:08:04  2018-04-25 13:20:10  2018-04-25 13:20:10   
1     2018-04-25 13:45:10  2018-04-26 00:56:16  2018-

In [46]:
processed_data_all=processed_data_all.set_index('time')

In [47]:
def missing(df):
    missing_number = df.isnull().sum().sort_values(ascending=False)
    missing_percent = (df.isnull().sum() / df.isnull().count()).sort_values(ascending=False)
    missing_values = pd.concat([missing_number, missing_percent], axis=1,keys=['missing_number','missing_percent'])
    total_missing_number=missing_number.sum()
    total_missing_percent=missing_percent.sum()
    total_row=pd.DataFrame([[total_missing_number,total_missing_percent]],
                           columns=['missing_number','missing_percent'],
                           index=['total'])
    missing_values=pd.concat([missing_values,total_row])
    return missing_values
# Define a function to show the number and percentage of missing values for each column, as well as the total number of missing values in the entire dataset.

In [48]:
missing(processed_data_all)

,missing_number,missing_percent
Unnamed: 0,0,0.0
id,0,0.0
windspeed,0,0.0
pressure,0,0.0
cloud_cover_description,0,0.0
cloud_cover,0,0.0
temperature,0,0.0
timestamp,0,0.0
userID,0,0.0
requestedDeparture,0,0.0


In [49]:
processed_data_all.head(25)

,Unnamed: 0,id,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,sessionID,siteID,spaceID,stationID,...,paymentRequired,requestedDeparture,userID,timestamp,temperature,cloud_cover,cloud_cover_description,pressure,windspeed,precipitation
time,,,,,,,,,,,,,,,,,,,,,
2018-04-25 11:08:04,0,5bc90cb9f9af8b0d7fe77cd2,2018-04-25 11:08:04,2018-04-25 13:20:10,2018-04-25 13:20:10,7.932,2_39_78_362_2018-04-25 11:08:04.400812,2,CA-496,2-39-78-362,...,unknown,unknown,unknown,2018-04-25 10:53:00,12.0,26.0,Cloudy,989.11,11.0,0.0
2018-04-25 13:45:10,1,5bc90cb9f9af8b0d7fe77cd3,2018-04-25 13:45:10,2018-04-26 00:56:16,2018-04-25 16:44:15,10.013,2_39_95_27_2018-04-25 13:45:09.617470,2,CA-319,2-39-95-27,...,unknown,unknown,unknown,2018-04-25 13:53:00,12.0,20.0,Fog,989.44,7.0,0.0
2018-04-25 13:45:50,2,5bc90cb9f9af8b0d7fe77cd4,2018-04-25 13:45:50,2018-04-25 23:04:45,2018-04-25 14:51:44,5.257,2_39_79_380_2018-04-25 13:45:49.962001,2,CA-489,2-39-79-380,...,unknown,unknown,unknown,2018-04-25 13:53:00,12.0,20.0,Fog,989.44,7.0,0.0
2018-04-25 14:37:06,3,5bc90cb9f9af8b0d7fe77cd5,2018-04-25 14:37:06,2018-04-25 23:55:34,2018-04-25 16:05:22,5.177,2_39_79_379_2018-04-25 14:37:06.460772,2,CA-327,2-39-79-379,...,unknown,unknown,unknown,2018-04-25 14:53:00,12.0,26.0,Cloudy,990.10,6.0,0.0
2018-04-25 14:40:34,4,5bc90cb9f9af8b0d7fe77cd6,2018-04-25 14:40:34,2018-04-25 23:03:12,2018-04-25 17:40:30,10.119,2_39_79_381_2018-04-25 14:40:33.638896,2,CA-490,2-39-79-381,...,unknown,unknown,unknown,2018-04-25 14:53:00,12.0,26.0,Cloudy,990.10,6.0,0.0
2018-04-25 14:43:50,5,5bc90cb9f9af8b0d7fe77cd7,2018-04-25 14:43:50,2018-04-26 01:17:30,2018-04-25 16:18:28,7.910,2_39_139_28_2018-04-25 14:43:49.647430,2,CA-303,2-39-139-28,...,unknown,unknown,unknown,2018-04-25 14:53:00,12.0,26.0,Cloudy,990.10,6.0,0.0
2018-04-25 14:47:42,6,5bc90cb9f9af8b0d7fe77cd8,2018-04-25 14:47:42,2018-04-25 18:27:51,2018-04-25 18:27:42,15.294,2_39_91_441_2018-04-25 14:47:41.776352,2,CA-499,2-39-91-441,...,unknown,unknown,unknown,2018-04-25 14:53:00,12.0,26.0,Cloudy,990.10,6.0,0.0
2018-04-25 14:58:25,7,5bc90cb9f9af8b0d7fe77cd9,2018-04-25 14:58:25,2018-04-25 19:06:29,2018-04-25 16:48:29,6.953,2_39_79_377_2018-04-25 14:58:25.255583,2,CA-325,2-39-79-377,...,unknown,unknown,unknown,2018-04-25 14:53:00,12.0,26.0,Cloudy,990.10,6.0,0.0
2018-04-25 15:10:52,8,5bc90cb9f9af8b0d7fe77cda,2018-04-25 15:10:52,2018-04-25 18:15:46,2018-04-25 16:07:56,2.174,2_39_79_382_2018-04-25 15:10:51.871020,2,CA-491,2-39-79-382,...,unknown,unknown,unknown,2018-04-25 15:05:00,13.0,21.0,Haze,990.10,6.0,0.0


In [50]:
processed_data_all = processed_data_all.drop(columns=['Unnamed: 0','timestamp','timezone'])

In [51]:
processed_data_all.to_csv('processed_data_all.csv', index=False)